# R1-02 — Per-path evaluation on ALL 20 IMUWiFine test paths (camera-ready, ICINCO 2026 paper #122)

Produces the data for the new Figure 8: per-path test MAE on **all 20 IMUWiFine floor-4 test
paths** for three methods:

1. **Ours** — loads the R1-01 fusion checkpoints from Drive (`runs/imuwifine_seed*/model_last.pt`),
   no retraining; all 3 seeds evaluated (figure uses seed 42, text quotes the seed spread).
2. **wlan_localization** — k-NN fingerprinting (k=3, distance-weighted), fit on train paths,
   evaluated per test path.
3. **IMUWiFine LSTM baseline** — one training run of the clean-room LSTM on this data build
   (old checkpoint lost with the workstation), then per-path test MAE.

Every result cell writes its JSON to Drive under `r1_02/` and **skips itself on re-run** if the
JSON is already there, so the notebook survives disconnects. Final cell prints the PASTE-BACK
JSON for the figure + Section 6.3 rewrite.

Use a GPU runtime (Runtime > Change runtime type > T4 GPU). Rough total: 30-45 min
(LSTM training dominates).


In [ ]:
# ==== Parameters ====
SEEDS = [42, 7, 123]        # fusion checkpoints to evaluate (from R1-01)
FIGURE_SEED = 42            # figures use this seed per the Sec 5.4 convention
EPOCHS = 40                 # paper config - kept so the rebuilt trainer matches R1-01 exactly
K = 4                       # n_instants, paper config
BATCH = 128                 # paper config
MBL = False                 # modality_balanced_loss off = paper config
DRIVE_DIR = "navlori_camera_ready"


In [ ]:
# ==== Google Drive ====
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
OUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_DIR
R102 = OUT_ROOT / "r1_02"
R102.mkdir(parents=True, exist_ok=True)
print("results root:", R102)
ckpts = sorted((OUT_ROOT / "runs").glob("imuwifine_seed*/model_last.pt"))
print("fusion checkpoints found:", [str(p.parent.name) for p in ckpts])
assert len(ckpts) >= 3, "R1-01 checkpoints missing from Drive - tell Claude"


In [ ]:
# ==== Clone repo + submodules + self-healing imports ====
import os, sys, subprocess, json, time, random, io, re
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (enable a GPU runtime!)")

REPO = Path("/content/navlori-fusion")
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/moebachar/navlori-fusion.git", str(REPO)])
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip(): print(out.strip()[-2000:])
    return r.returncode

# ronin + dpvo needed by eager imports in src.pipeline.encoders/baselines;
# wlan_localization needed for the k-NN baseline of this notebook.
SUBMODULES = {
    "external_methods/ronin": ("https://github.com/Sachini/ronin",
                               "805b7f0f28bb164ce89ada9ac05a9470dbe3d715",
                               "source/model_resnet1d.py"),
    "external_methods/dpvo": ("https://github.com/princeton-vl/DPVO",
                              "859bbbfdac6c6185f345003b3c473901fcd13ace",
                              "dpvo/extractor.py"),
    "external_methods/wlan_localization": ("https://github.com/sharan-naribole/wlan_localization",
                              "5e1949dac00b779268eca26b13081e9ee901c47e",
                              "src/wlan_localization/models/position_regressor.py"),
}
print("-- git submodule update --init --")
_run(["git", "submodule", "update", "--init"] + list(SUBMODULES), cwd=str(REPO))
import shutil
for path, (url, sha, marker) in SUBMODULES.items():
    d = REPO / path
    if not (d / marker).exists():
        print(f"-- submodule route did not materialize {path}; direct clone fallback --")
        if d.exists():
            shutil.rmtree(d, ignore_errors=True)
        _run(["git", "clone", url, str(d)])
        _run(["git", "checkout", sha], cwd=str(d))
    assert (d / marker).exists(), f"{path}/{marker} STILL missing - save a copy to GitHub and tell Claude"
    print(f"OK: {path} ({marker} present)")

PIPNAME = {"omegaconf": "omegaconf", "yaml": "pyyaml", "sklearn": "scikit-learn",
           "cv2": "opencv-python-headless", "mlflow": "mlflow", "torchdiffeq": "torchdiffeq",
           "seaborn": "seaborn", "influxdb_client": "influxdb-client", "plotly": "plotly",
           "quaternion": "numpy-quaternion", "PIL": "pillow", "skimage": "scikit-image"}
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "quaternion"],
               capture_output=True)
_tried = set()
for attempt in range(10):
    try:
        from src.pipeline.fusion.builder import (build_datamodule, build_encoders,
                                                 build_model, build_trainer, load_config)
        print("imports OK"); break
    except ModuleNotFoundError as e:
        pkg = e.name.split(".")[0]
        if pkg in _tried:
            raise RuntimeError(
                f"module '{e.name}' still missing after installing '{PIPNAME.get(pkg, pkg)}' - "
                "save a copy to GitHub and tell Claude.") from e
        _tried.add(pkg)
        pip = PIPNAME.get(pkg, pkg)
        print("missing module:", e.name, "-> pip install", pip)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip])
        if r.returncode != 0:
            raise RuntimeError(
                f"'{e.name}' is not pip-installable - likely a repo-local module. "
                "Save a copy of this notebook to GitHub and tell Claude.") from e
else:
    raise RuntimeError("imports still failing after installs - send Claude the error above")

import numpy as np

def set_global_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## Data — IMUWiFine floor 4

Restored from the Drive cache left by the R1-01 notebook; the HuggingFace download +
conversion below only runs if the cache is somehow gone.


In [ ]:
# ==== IMUWiFine floor 4: restore from Drive cache (HF fallback) ====
import shutil
DATA_CACHE = OUT_ROOT / "data_cache"

def run_logged(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip(): print(out.strip()[-3000:])
    return r.returncode

def drive_restore(name, marker="metadata.json"):
    dst = REPO / "data" / name
    if (dst / marker).exists(): return "already in session"
    src = DATA_CACHE / name
    if (src / marker).exists():
        shutil.copytree(src, dst, dirs_exist_ok=True); return "restored from Drive cache"
    return None

st = drive_restore("imuwifine_floor4")
if st:
    print("IMUWiFine:", st)
else:
    from huggingface_hub import snapshot_download
    raw = Path(snapshot_download(repo_id="issai/IMUWiFine", repo_type="dataset",
                                 local_dir="/content/IMUWiFine_raw"))
    hits = [p for p in raw.rglob("raw_IUMIWiFi") if p.is_dir()]
    raw_root = None
    if hits:
        p = hits[0]
        if p.parent.name == "IMU_DATA":
            raw_root = p.parent.parent
        else:
            stage = Path("/content/iw_root")
            (stage / "IMU_DATA").mkdir(parents=True, exist_ok=True)
            link = stage / "IMU_DATA" / "raw_IUMIWiFi"
            if not link.exists(): link.symlink_to(p)
            tests = [t for t in raw.rglob("test") if t.is_dir() and list(t.glob("test_*.txt"))]
            if tests:
                tl = stage / "IMU_DATA" / "test"
                if not tl.exists(): tl.symlink_to(tests[0])
            raw_root = stage
    ok = raw_root and (Path(raw_root) / "IMU_DATA" / "raw_IUMIWiFi" / "4th_floor").exists() \
         and (Path(raw_root) / "IMU_DATA" / "test").exists()
    if not ok:
        raise RuntimeError("unexpected IMUWiFine layout - tell Claude")
    if run_logged([sys.executable, "scripts/convert_imuwifine.py", "--floor", "4",
                   "--raw-root", str(raw_root), "--out-root", "data"]) != 0:
        raise RuntimeError("convert_imuwifine.py failed - output above")
n_paths = len(list((REPO / "data" / "imuwifine_floor4").glob("path_*")))
print(f"IMUWiFine floor 4 ready: {n_paths} paths (expected 80: 40 train / 20 val / 20 test)")


## Method 1 — Ours: per-path MAE from the R1-01 checkpoints (no retraining)

Rebuilds the exact R1-01 trainer (same config overrides), loads each seed's
`model_last.pt`, predicts the test split, and groups errors by path. The overall
MAE is cross-checked against the R1-01 `summary.json` for that seed.


In [ ]:
# ==== Ours per-path (3 seeds) ====
for seed in SEEDS:
    out_p = R102 / f"ours_seed{seed}.json"
    if out_p.exists():
        d = json.loads(out_p.read_text())
        print(f"skip ours seed {seed}: test {d['overall']['test_mae_m']:.2f} m "
              f"({len(d['per_path_test'])} paths)")
        continue
    print(f"\n===== ours seed {seed} =====", flush=True)
    set_global_seed(seed)
    cfg = load_config("imuwifine")
    cfg.temporal.n_instants = K
    cfg.train.modality_balanced_loss = MBL
    cfg.data.batch_size = BATCH
    dm = build_datamodule(cfg)
    encs, vision = build_encoders(cfg, dm)
    model = build_model(cfg, encs)
    trainer = build_trainer(cfg, model, dm, run_dir=f"/content/tmp_eval_seed{seed}")
    ckpt = OUT_ROOT / "runs" / f"imuwifine_seed{seed}" / "model_last.pt"
    sd = torch.load(ckpt, map_location=trainer.device)
    trainer.model.load_state_dict(sd)
    print("checkpoint loaded:", ckpt.name)

    res = {"seed": seed, "overall": {}, "per_path_test": {}}
    preds_v, tgts_v = trainer.predict("val")
    res["overall"]["val_mae_m"] = float((preds_v - tgts_v).norm(dim=1).mean())
    preds_t, tgts_t = trainer.predict("test")
    res["overall"]["test_mae_m"] = float((preds_t - tgts_t).norm(dim=1).mean())

    summ = json.loads((OUT_ROOT / "runs" / f"imuwifine_seed{seed}" / "summary.json").read_text())
    dv = abs(res["overall"]["val_mae_m"] - summ["val_mae_m"])
    dt = abs(res["overall"]["test_mae_m"] - summ["test_mae_m"])
    print(f"overall: val {res['overall']['val_mae_m']:.3f} m (R1-01: {summ['val_mae_m']:.3f}), "
          f"test {res['overall']['test_mae_m']:.3f} m (R1-01: {summ['test_mae_m']:.3f})")
    if max(dv, dt) > 0.05:
        print("!! WARNING: overall MAE deviates from the R1-01 summary by "
              f"{max(dv, dt):.3f} m - tell Claude before using these numbers")

    err = (preds_t - tgts_t).norm(dim=1).numpy()
    ds_test = trainer.dm.test_ds
    pids = np.array([r["path_id"] for r in ds_test._gt_rows])[:len(err)]
    for pid in sorted(np.unique(pids)):
        m = pids == pid
        res["per_path_test"][str(int(pid))] = {
            "mae": float(err[m].mean()), "n": int(m.sum())}
        print(f"  path_{int(pid):02d}: MAE {err[m].mean():.3f} m  n={int(m.sum())}")
    out_p.write_text(json.dumps(res, indent=2))
    print("wrote", out_p.name)
    del trainer, model, dm, encs
    torch.cuda.empty_cache()
print("\nours: done")


## Method 2 — wlan_localization: per-path k-NN

Same protocol as the paper's overall IMUWiFine wlanloc numbers
(`scripts/_eval_wlanloc_imuwifine.py`): fit Box-Cox + PCA preprocessor and the
k-NN regressor (k=3, manhattan, distance-weighted) on the 40 train paths, then
evaluate each of the 20 test paths separately. Overall val/test are printed as a
cross-check against the script's whole-split numbers.


In [ ]:
# ==== wlanloc per-path ====
out_p = R102 / "wlanloc_perpath.json"
if out_p.exists():
    d = json.loads(out_p.read_text())
    print(f"skip wlanloc: test {d['overall']['test_mae_m']:.2f} m "
          f"({len(d['per_path_test'])} paths)")
else:
    import importlib.util
    spec = importlib.util.spec_from_file_location(
        "eval_wlanloc_iwf", REPO / "scripts" / "_eval_wlanloc_imuwifine.py")
    wl = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(wl)
    wl.DATA = REPO / "data" / "imuwifine_floor4"

    from src.pipeline.baselines import load_position_regressor, load_preprocessor
    print("Loading splits...", flush=True)
    Xtr_raw, Ytr, master = wl.load_split_rssi(wl.TRAIN_PATHS)
    Xva_raw, Yva, _ = wl.load_split_rssi(wl.VAL_PATHS, master)
    print(f"train {len(Xtr_raw)}  val {len(Xva_raw)}  APs {Xtr_raw.shape[1]}")

    # per-path test matrices, same master AP columns
    test_X, test_Y = {}, {}
    for pid in wl.TEST_PATHS:
        X, Y, _ = wl.load_split_rssi([pid], master)
        if len(X): test_X[pid], test_Y[pid] = X, Y
    print(f"test paths loaded: {len(test_X)}")

    # the vendored convention: undetected = +100 before the preprocessor
    def to_wlanloc(X): return np.where(X == -100.0, 100.0, X)
    pre = load_preprocessor()()
    use_affine = False
    try:
        Xtr = pre.fit_transform(to_wlanloc(Xtr_raw))
        Xva = pre.transform(to_wlanloc(Xva_raw))
        tX = {p: pre.transform(to_wlanloc(X)) for p, X in test_X.items()}
    except Exception as e:
        print(f"Preprocessor failed ({type(e).__name__}: {e}); affine fallback")
        use_affine = True
        Xtr = (Xtr_raw + 100.0) / 100.0
        Xva = (Xva_raw + 100.0) / 100.0
        tX = {p: (X + 100.0) / 100.0 for p, X in test_X.items()}

    reg = load_position_regressor()(k=3, metric="manhattan", weights="distance")
    reg.fit_location(0, 0, Xtr, Ytr)
    knn = reg.models[(0, 0)]

    res = {"overall": {}, "per_path_test": {},
           "preprocessor": "affine fallback" if use_affine else "Box-Cox + PCA (fit on train)"}
    ev = np.sqrt(((knn.predict(Xva) - Yva) ** 2).sum(1))
    res["overall"]["val_mae_m"] = float(ev.mean())
    all_te = []
    for pid in sorted(tX):
        e = np.sqrt(((knn.predict(tX[pid]) - test_Y[pid]) ** 2).sum(1))
        all_te.append(e)
        res["per_path_test"][str(pid)] = {"mae": float(e.mean()), "n": int(len(e))}
        print(f"  path_{pid:02d}: MAE {e.mean():.3f} m  n={len(e)}")
    res["overall"]["test_mae_m"] = float(np.concatenate(all_te).mean())
    print(f"overall: val {res['overall']['val_mae_m']:.3f} m  "
          f"test {res['overall']['test_mae_m']:.3f} m")
    out_p.write_text(json.dumps(res, indent=2))
    print("wrote", out_p.name)


## Method 3 — IMUWiFine LSTM baseline: train once + per-path MAE

Runs the repo's one-shot script (`scripts/_train_imuwifine_baseline_on_iwfine.py`)
as a module with its data root pointed at this session's build: trains the clean-room
4-layer LSTM (seed 42 fixed inside the script, 40 epochs, best-val checkpointing) and
saves per-path test predictions. Takes roughly 10-25 min on a T4.


In [ ]:
# ==== LSTM baseline ====
out_p = R102 / "lstm_perpath.json"
if out_p.exists():
    d = json.loads(out_p.read_text())
    print(f"skip lstm: best val {d['best_val_mae_m']:.2f} m "
          f"({len(d['per_path_test'])} paths)")
else:
    import importlib.util
    spec = importlib.util.spec_from_file_location(
        "train_lstm_iwf", REPO / "scripts" / "_train_imuwifine_baseline_on_iwfine.py")
    tl = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(tl)
    tl.ROOT = REPO                                       # save_dir under the repo
    tl.IWF_ROOT = REPO / "data" / "imuwifine_floor4"     # this session's data build
    t0 = time.time()
    tl.main()
    print(f"LSTM done in {(time.time()-t0)/60:.1f} min")

    save_dir = REPO / "runs" / "main_table" / "imuwifine" / "imuwifine_baseline"
    per_path_raw = json.loads((save_dir / "per_path_test.json").read_text())
    ckpt = torch.load(save_dir / "model.pt", map_location="cpu", weights_only=False)
    res = {"seed": 42, "best_val_mae_m": float(ckpt["best_val_mae"]),
           "per_path_test": {str(k): {"mae": v["mae"], "n": v["n"]}
                             for k, v in per_path_raw.items()}}
    maes = [v["mae"] for v in res["per_path_test"].values()]
    ns = [v["n"] for v in res["per_path_test"].values()]
    res["overall_test_mae_m"] = float(np.average(maes, weights=ns))
    out_p.write_text(json.dumps(res, indent=2))
    shutil.copy(save_dir / "model.pt", R102 / "lstm_model.pt")
    print("wrote", out_p.name, "+ lstm_model.pt to Drive")


## Aggregate — the Figure 8 data (PASTE-BACK)

One row per test path, three methods. `ours` carries seed 42 (the figure value) and the
mean and standard deviation over the 3 seeds. Macro-average = unweighted mean over the 20
per-path MAEs; overall = sample-weighted. The 4 paths of the submitted Figure 8 are marked.


In [ ]:
# ==== Aggregate + PASTE-BACK JSON ====
import statistics
ours = {s: json.loads((R102 / f"ours_seed{s}.json").read_text()) for s in SEEDS}
wlan = json.loads((R102 / "wlanloc_perpath.json").read_text())
lstm = json.loads((R102 / "lstm_perpath.json").read_text())

pids = sorted(int(p) for p in ours[FIGURE_SEED]["per_path_test"])
OLD_FIG_PATHS = [64, 68, 69, 71]  # the submitted Figure 8 subset

table = {}
for pid in pids:
    k = str(pid)
    o = [ours[s]["per_path_test"][k]["mae"] for s in SEEDS]
    table[k] = {
        "wlanloc": round(wlan["per_path_test"][k]["mae"], 3) if k in wlan["per_path_test"] else None,
        "lstm": round(lstm["per_path_test"][k]["mae"], 3) if k in lstm["per_path_test"] else None,
        "ours_seed42": round(ours[FIGURE_SEED]["per_path_test"][k]["mae"], 3),
        "ours_mean": round(statistics.mean(o), 3),
        "ours_std": round(statistics.stdev(o), 3),
        "n": ours[FIGURE_SEED]["per_path_test"][k]["n"],
        "in_old_figure": pid in OLD_FIG_PATHS,
    }

def macro(vals): return round(statistics.mean([v for v in vals if v is not None]), 3)
agg = {
    "comment": "R1-02 per-path test MAE (m), IMUWiFine floor 4, all 20 test paths",
    "figure_seed": FIGURE_SEED,
    "per_path": table,
    "macro_avg": {
        "wlanloc": macro([r["wlanloc"] for r in table.values()]),
        "lstm": macro([r["lstm"] for r in table.values()]),
        "ours_seed42": macro([r["ours_seed42"] for r in table.values()]),
        "ours_mean_of_means": macro([r["ours_mean"] for r in table.values()]),
    },
    "overall": {
        "wlanloc": {"val": round(wlan["overall"]["val_mae_m"], 3),
                    "test": round(wlan["overall"]["test_mae_m"], 3)},
        "lstm": {"best_val": round(lstm["best_val_mae_m"], 3),
                 "test": round(lstm["overall_test_mae_m"], 3)},
        "ours": {s: {"val": round(ours[s]["overall"]["val_mae_m"], 3),
                     "test": round(ours[s]["overall"]["test_mae_m"], 3)} for s in SEEDS},
    },
    "wins": {
        "ours_best_seed42": sum(1 for r in table.values()
                                if r["lstm"] is not None and r["wlanloc"] is not None
                                and r["ours_seed42"] <= min(r["lstm"], r["wlanloc"])),
        "lstm_best": sum(1 for r in table.values()
                         if r["lstm"] is not None and r["wlanloc"] is not None
                         and r["lstm"] < min(r["ours_seed42"], r["wlanloc"])),
    },
}

hdr = f"{'path':>5} {'wlanloc':>8} {'lstm':>7} {'ours42':>7} {'ours m+/-s':>12}  {'old fig':>7}"
print(hdr); print("-" * len(hdr))
for pid in pids:
    r = table[str(pid)]
    print(f"{pid:>5} {str(r['wlanloc']):>8} {str(r['lstm']):>7} {r['ours_seed42']:>7} "
          f"{r['ours_mean']:>6}+/-{r['ours_std']:<4}  {'*' if r['in_old_figure'] else '':>7}")
print("-" * len(hdr))
print("macro:", agg["macro_avg"])
print("wins :", agg["wins"])

(R102 / "r1_02_perpath.json").write_text(json.dumps(agg, indent=2))
print("\nwrote r1_02_perpath.json to Drive\n")
print("=" * 30, "PASTE-BACK JSON", "=" * 30)
print(json.dumps(agg))
